# Notebook 02 — EDA & SQL Analysis

**Goal**: Explore the data entirely through SQL queries, then visualize findings.
This notebook is the **SQL showcase** — every query demonstrates a real skill.

**Queries covered**:
1. Hourly demand overview
2. Weather impact on trips (JOIN + CASE + PERCENTILE_CONT)
3. Event-driven demand deviation (CTE + window functions)
4. Zone ranking (DENSE_RANK)
5. Hour × day-of-week heatmap
6. Month-over-month growth (LAG)
7. Precipitation threshold analysis
8. Top origin-destination pairs

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sqlalchemy import text
from src.utils.db import get_engine

engine = get_engine()

def sql(query: str) -> pd.DataFrame:
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn)

print('Ready.')

## Query 1 — Hourly demand overview

In [ ]:
df_hourly = sql("""
    SELECT
        date_trunc('hour', pickup_dt)           AS hour_bucket,
        COUNT(*)                                 AS trip_count,
        ROUND(AVG(fare_amount)::NUMERIC, 2)      AS avg_fare,
        ROUND(SUM(total_amount)::NUMERIC, 2)     AS revenue
    FROM taxi_trips
    GROUP BY 1
    ORDER BY 1
""")
df_hourly['hour_bucket'] = pd.to_datetime(df_hourly['hour_bucket'])

fig = px.line(df_hourly, x='hour_bucket', y='trip_count',
              title='NYC Taxi Hourly Demand (Jan–Jun 2024)',
              labels={'hour_bucket': 'Hour', 'trip_count': 'Trips'})
fig.update_traces(line_color='#3498db', line_width=1)
fig.show()

print(f'Total trips: {df_hourly["trip_count"].sum():,}')
print(f'Total revenue: ${df_hourly["revenue"].sum():,.0f}')

## Query 2 — Weather impact (JOIN + CASE + PERCENTILE_CONT)

In [ ]:
df_weather = sql("""
    SELECT
        CASE
            WHEN w.weather_code IN (0,1)          THEN 'Clear'
            WHEN w.weather_code IN (2,3)          THEN 'Cloudy'
            WHEN w.weather_code BETWEEN 51 AND 67 THEN 'Rain'
            WHEN w.weather_code BETWEEN 71 AND 77 THEN 'Snow'
            ELSE 'Other'
        END                                           AS weather_category,
        COUNT(t.trip_id)                              AS total_trips,
        ROUND(AVG(t.fare_amount)::NUMERIC, 2)         AS avg_fare,
        PERCENTILE_CONT(0.5)
            WITHIN GROUP (ORDER BY t.fare_amount)     AS median_fare
    FROM taxi_trips t
    JOIN weather_hourly w
        ON date_trunc('hour', t.pickup_dt) = w.obs_dt
    GROUP BY weather_category
    ORDER BY total_trips DESC
""")

fig = px.bar(df_weather, x='weather_category', y='total_trips',
             color='avg_fare', color_continuous_scale='RdYlGn',
             title='Trip Volume & Average Fare by Weather Condition',
             text_auto='.0f')
fig.show()
df_weather

## Query 3 — Event demand deviation (CTE + window functions)

In [ ]:
df_events = sql("""
    WITH daily_demand AS (
        SELECT DATE(pickup_dt) AS trip_date, COUNT(*) AS daily_trips
        FROM taxi_trips
        GROUP BY 1
    ),
    windowed AS (
        SELECT
            trip_date,
            daily_trips,
            ROUND(AVG(daily_trips) OVER (
                ORDER BY trip_date
                ROWS BETWEEN 7 PRECEDING AND 7 FOLLOWING
            )::NUMERIC, 0) AS rolling_avg_14d
        FROM daily_demand
    )
    SELECT
        w.trip_date,
        w.daily_trips,
        w.rolling_avg_14d,
        e.event_name,
        e.event_type,
        ROUND(
            (w.daily_trips - w.rolling_avg_14d)
            / NULLIF(w.rolling_avg_14d, 0) * 100
        , 1) AS pct_deviation
    FROM windowed w
    LEFT JOIN events e ON w.trip_date = e.event_date
    WHERE e.event_name IS NOT NULL
    ORDER BY pct_deviation DESC
    LIMIT 15
""")

fig = px.bar(df_events.sort_values('pct_deviation'),
             x='pct_deviation', y='event_name', orientation='h',
             color='pct_deviation', color_continuous_scale='RdYlGn',
             title='Top Event Impact: % Demand Deviation from 14-Day Rolling Avg',
             labels={'pct_deviation': '% Deviation', 'event_name': 'Event'})
fig.show()
df_events

## Query 4 — Hour × Day-of-Week heatmap

In [ ]:
df_heatmap = sql("""
    SELECT
        EXTRACT(DOW FROM pickup_dt)::INTEGER  AS day_of_week,
        EXTRACT(HOUR FROM pickup_dt)::INTEGER AS hour_of_day,
        COUNT(*) AS trip_count
    FROM taxi_trips
    GROUP BY 1, 2
    ORDER BY 1, 2
""")

pivot = df_heatmap.pivot(index='day_of_week', columns='hour_of_day', values='trip_count')
day_labels = ['Sun','Mon','Tue','Wed','Thu','Fri','Sat']

fig = go.Figure(data=go.Heatmap(
    z=pivot.values,
    x=[f'{h}:00' for h in range(24)],
    y=day_labels,
    colorscale='Blues',
    text=pivot.values,
    texttemplate='%{text:.0f}',
))
fig.update_layout(
    title='Demand Heatmap: Hour × Day of Week',
    xaxis_title='Hour of Day',
    yaxis_title='Day of Week',
)
fig.show()

## Query 5 — Month-over-month growth (LAG window function)

In [ ]:
df_mom = sql("""
    WITH monthly AS (
        SELECT
            date_trunc('month', pickup_dt)          AS month,
            COUNT(*)                                 AS monthly_trips,
            ROUND(SUM(total_amount)::NUMERIC, 2)     AS monthly_revenue
        FROM taxi_trips
        GROUP BY 1
    )
    SELECT
        month,
        monthly_trips,
        monthly_revenue,
        LAG(monthly_trips) OVER (ORDER BY month) AS prev_trips,
        ROUND(
            (monthly_trips - LAG(monthly_trips) OVER (ORDER BY month))::NUMERIC
            / NULLIF(LAG(monthly_trips) OVER (ORDER BY month), 0) * 100
        , 1) AS mom_growth_pct
    FROM monthly
    ORDER BY month
""")

df_mom['month'] = pd.to_datetime(df_mom['month'])

fig = px.bar(df_mom, x='month', y='mom_growth_pct',
             title='Month-over-Month Trip Growth (%)',
             color='mom_growth_pct', color_continuous_scale='RdYlGn',
             labels={'month': 'Month', 'mom_growth_pct': 'MoM Growth (%)'})
fig.show()
df_mom